<a href="https://colab.research.google.com/github/LikhithR916/AI_Lab/blob/main/1BM24CS147_week3__A_Search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import heapq

class PuzzleNode:
    def __init__(self, state, goal_state, parent=None, move=None, g=0):
        self.state = state            # Current board configuration (1D tuple)
        self.goal_state = goal_state  # Targeted goal configuration (1D tuple)
        self.parent = parent          # Link to parent node
        self.move = move              # Direction moved to reach this state
        self.g = g                    # g(n): Exact cost/moves from start node
        self.h = self.calculate_misplaced_tiles()  # h(n): Misplaced tiles heuristic
        self.f = self.g + self.h                   # f(n) = g(n) + h(n)

    def calculate_misplaced_tiles(self):
        """Calculates h(n) by counting tiles that are out of place."""
        misplaced = 0
        for i in range(9):
            tile = self.state[i]
            if tile != 0 and tile != self.goal_state[i]:
                misplaced += 1
        return misplaced

    def get_neighbors(self):
        """Generates all valid adjacent states by moving the blank slot (0)."""
        neighbors = []
        blank_idx = self.state.index(0)
        row, col = divmod(blank_idx, 3)

        moves = {
            'Up': (-1, 0),
            'Down': (1, 0),
            'Left': (0, -1),
            'Right': (0, 1)
        }

        for move_name, (dr, dc) in moves.items():
            new_row, new_col = row + dr, col + dc

            if 0 <= new_row < 3 and 0 <= new_col < 3:
                new_blank_idx = new_row * 3 + new_col
                new_state = list(self.state)
                new_state[blank_idx], new_state[new_blank_idx] = new_state[new_blank_idx], new_state[blank_idx]

                neighbors.append(PuzzleNode(tuple(new_state), self.goal_state, parent=self, move=move_name, g=self.g + 1))

        return neighbors

    def __lt__(self, other):
        # Tie-breaker: choose the one with lowest f(n)
        return self.f < other.f

def solve_8_puzzle_with_logs(start_state, goal_state):
    """Solves the 8-puzzle and prints out every explored state step-by-step."""
    open_set = []
    start_node = PuzzleNode(start_state, goal_state)
    heapq.heappush(open_set, (start_node.f, start_node))

    closed_set = set()
    exploration_step = 1

    print("\n" + "="*40)
    print("      STARTING A* STATE EXPLORATION")
    print("="*40)

    while open_set:
        # Pop the state with the lowest f(n) value
        _, current_node = heapq.heappop(open_set)

        # Print metrics for the state currently being explored
        print(f"\n[Explored Node #{exploration_step}]")
        if current_node.move:
            print(f"Action : Move {current_node.move}")
        else:
            print("Action taken to get here: Initial Configuration")

        print(f"Metrics: g(n) = {current_node.g}, h(n) = {current_node.h}, f(n) = {current_node.f}")
        print("Board Layout:")
        print_board(current_node.state)

        exploration_step += 1

        # Base case: Goal reached
        if current_node.state == goal_state:
            print("\n*** GOAL STATE REACHED! ***")
            return reconstruct_path(current_node)

        closed_set.add(current_node.state)

        for neighbor in current_node.get_neighbors():
            if neighbor.state in closed_set:
                continue

            heapq.heappush(open_set, (neighbor.f, neighbor))

    return None  # Unsolvable puzzle

def reconstruct_path(node):
    """Traces back from the goal node to compile the strict final path."""
    path = []
    current = node
    while current.parent is not None:
        path.append((current.move, current.state, current.g, current.h, current.f))
        current = current.parent
    path.reverse()
    return path

def print_board(state):
    """Pretty prints the 1D state array into a 3x3 layout."""
    for i in range(0, 9, 3):
        print(f"  {state[i]}  {state[i+1]}  {state[i+2]}")
    print("  ---------")

def get_matrix_input(prompt_title):
    """Prompts user to input a 3x3 layout row by row."""
    print(f"\n{prompt_title} (Use 0 for blank spacer):")
    matrix_elements = []

    for row in range(3):
        while True:
            try:
                row_input = input(f"Enter Row {row + 1} elements (3 space-separated numbers): ")
                parsed_row = [int(x) for x in row_input.split()]
                if len(parsed_row) != 3:
                    print("Error: A row must have exactly 3 values. Try again.")
                    continue
                matrix_elements.extend(parsed_row)
                break
            except ValueError:
                print("Error: Enter integer inputs only.")

    return tuple(matrix_elements)

def validate_states(initial, goal):
    """Validates elements consistency across both inputs."""
    valid_set = set(range(9))
    if set(initial) != valid_set or set(goal) != valid_set:
        print("\n[Validation Error]: Both inputs must contain distinct numbers from 0 to 8.")
        return False
    return True

# --- EXECUTION ---
if __name__ == "__main__":
    print("=== 8-PUZZLE MATRIX INTERACTIVE SOLVER ===")

    # 1. Take Matrix inputs row-by-row
    initial_puzzle = get_matrix_input("--- INITIAL MATRIX ---")
    goal_puzzle = get_matrix_input("--- GOAL MATRIX ---")

    # 2. Check for layout structural validity
    if validate_states(initial_puzzle, goal_puzzle):
        # 3. Solve while generating verbose evaluation reports
        solution_steps = solve_8_puzzle_with_logs(initial_puzzle, goal_puzzle)

        if solution_steps:
            print(f"\nExecution Finished! Total moves in optimal solution path: {len(solution_steps)}")
        else:
            print("\nThis specific puzzle combination is mathematically unsolvable.")


=== 8-PUZZLE MATRIX INTERACTIVE SOLVER ===

--- INITIAL MATRIX --- (Use 0 for blank spacer):
Enter Row 1 elements (3 space-separated numbers): 2 8 3
Enter Row 2 elements (3 space-separated numbers): 1 6 4
Enter Row 3 elements (3 space-separated numbers): 7 0 5

--- GOAL MATRIX --- (Use 0 for blank spacer):
Enter Row 1 elements (3 space-separated numbers): 1 2 3
Enter Row 2 elements (3 space-separated numbers): 8 0 4
Enter Row 3 elements (3 space-separated numbers): 7 6 5

      STARTING A* STATE EXPLORATION

[Explored Node #1]
Action taken to get here: Initial Configuration
Metrics: g(n) = 0, h(n) = 4, f(n) = 4
Board Layout:
  2  8  3
  1  6  4
  7  0  5
  ---------

[Explored Node #2]
Action : Move Up
Metrics: g(n) = 1, h(n) = 3, f(n) = 4
Board Layout:
  2  8  3
  1  0  4
  7  6  5
  ---------

[Explored Node #3]
Action : Move Up
Metrics: g(n) = 2, h(n) = 3, f(n) = 5
Board Layout:
  2  0  3
  1  8  4
  7  6  5
  ---------

[Explored Node #4]
Action : Move Left
Metrics: g(n) = 2, h(n) 